# control

> A local socket a detached job runner listens on, so a host's page can prompt, steer, stop and answer its agent from another process.

In [ ]:
#| default_exp control

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail

In [ ]:
#| export
import json, os, re, socket, socketserver, stat, tempfile, threading, time
from dataclasses import asdict
from pathlib import Path
from fastcore.basics import patch
from shalya import acts, err
from ramabana.core import agent_err
from ramabana.agent import TurnResult, model_title
from ramabana.peers import (CONTROL, Envelope, _atomic_write, _rows, all_peers, answer, control_call, guard_of, is_partner, limit_refusal, next_hops, peer_lines, peer_text,
                            peers_dir, take, together_off, together_state, tell as send)
from ramabana.peer_report import save_report

## The socket

A program that hosts agent jobs, such as a web app, often runs each job in a process of its own. Restarting the web server then does not kill a turn halfway through. The page that started the job needs a way to reach it: to send a prompt, steer the running turn, stop it, and answer the approval requests it raises. `Control` provides that way. It opens a Unix socket in the session's run folder. A Unix socket is an address in the file system that only processes on this machine can connect to. Only the current user may connect.

The protocol is one JSON object per line in each direction. A request is `{"id", "method", "params"}`. A successful reply is `{"id", "result"}`. A failed reply is `{"id", "error"}`. `METHODS` lists the methods:

- `prompt(text)` starts a turn on a worker thread. If a turn is already running, `text` is queued to run after it.
- `steer(text)` gives `text` to the running turn with its next tool result. This redirects the model without stopping the turn.
- `tell(run_id, text)` queues `text` for the running sub-agent `run_id`, as `Agent.tell` does, and returns what it said: `told RUN`, or why nothing is listening. A transcript pane uses it for the lines typed into it.
- `cancel()` stops the running turn and drops the prompts queued behind it. The reply includes the number dropped as `dropped`.
- `asks()` lists the approval requests waiting for an answer, oldest first. `answer(id, ok, note)` answers one of them.
- `status()` returns `running`, `queued` (the number of queued prompts), `session` and `last`. `last` is how the last turn ended, or `None` before any turn has ended. It also returns what a peer lists: `name`, the session's name for a person; `state`, one of `idle`, `working` and `waiting for approval`; `folder`, its first open folder; and `at`, when it was last active.
- `events(offset)` returns `events`, the records in the session's event file from byte `offset` on, and `offset`, the value to pass next time. It returns whole lines only, at most `EVENTS_MAX` bytes per call. When the file has rolled over since the last read, it starts from byte 0 and sets `rotated` to true. It raises an error until the runner has called `agent.events_to(path)`.

`control_call(path, method, **params)` is the other end. It connects, sends one request and returns the `result`. On an error reply it raises `RuntimeError` with the error's message. Its `timeout` argument is 30 seconds by default. The page uses it, and any script that needs to reach a running job can use it.

`start` refuses a path where another runner is listening and a path that holds anything other than a socket. It removes a socket that a crashed runner left behind.

A Unix socket path takes at most about 100 bytes (`SOCKET_MAX`). When you give no path and the session's run folder is deeper, the socket goes in the temporary folder as `ramabana-<session>.sock`. An explicit `path` that is too long raises `ValueError`.

| Constant | Value | Limit |
|---|---|---|
| `SOCKET_MAX` | 100 | Bytes of a socket path. macOS allows 104. |
| `REQUEST_MAX` | 1,000,000 | Bytes of one request line |
| `EVENTS_MAX` | 1,000,000 | Bytes of the event file that one `events` call returns |

In [ ]:
#| export
METHODS = ('prompt', 'steer', 'tell', 'cancel', 'asks', 'answer', 'status', 'events', 'message')
SOCKET_MAX = 100   #: bytes a Unix socket path may take; macOS allows 104
REQUEST_MAX = 1_000_000   #: bytes one request line may take
EVENTS_MAX = 1_000_000    #: bytes of the event file one `events` call returns
EVENT_FILE = 'events.jsonl'   #: in a session's run folder: the event file a terminal session writes when nothing else does

def control_path(agent):
    "Where `agent`'s session listens: `CONTROL` in its run folder, or `ramabana-<session>.sock` in the temporary folder when that path is too long for a socket."
    p = Path(agent.runs_dir)/CONTROL
    return p if len(str(p).encode()) <= SOCKET_MAX else Path(tempfile.gettempdir())/f'ramabana-{agent.session_id}.sock'

def ago(t):
    "How long ago `t` was: `5m`, `3h` or `2d`."
    s = max(0, time.time() - (t or 0))
    return f'{int(s // 60)}m' if s < 3600 else f'{int(s // 3600)}h' if s < 86400 else f'{int(s // 86400)}d'

def pick_row(rows, choice):
    "The row numbered `choice` in `rows` as they were listed from 1, or None when `choice` is not one of those numbers."
    return rows[int(choice) - 1] if str(choice).isdigit() and 1 <= int(choice) <= len(rows) else None

def session_name(agent, at):
    "A live session's name for a person, titled as `Agent.sessions` titles it for `/resume`, else by its first folder's name, then its model and how long ago it was `at`."
    m = agent.session_meta()   # the stored index row: `sessions()` would rebuild the whole index on every `status`
    title = (m['title'] if m.get('manual') else model_title(m['title'])) or m.get('first_title', '')
    roots = list(agent.host.roots)
    return f"{title or (Path(roots[0]).name or roots[0] if roots else agent.session_id)} · {agent.model.name} · {ago(at)} ago"

class Control:
    "A Unix socket through which another process prompts, steers, stops and answers an agent running in this one."
    def __init__(self, agent,
                 path=None):   # the socket; None puts it in the session's run folder as `CONTROL`
        self.agent, self.last = agent, None
        self.path = Path(path) if path else control_path(agent)
        if len(str(self.path).encode()) > SOCKET_MAX: raise ValueError(f'a Unix socket path is at most {SOCKET_MAX} bytes: {self.path}')
        self.at = time.time()   #: when this session last started or finished a turn
        self._lock, self._running, self._server, self._inode = threading.Lock(), False, None, None
        self.report = None   #: the report saved when a work together mode ended, until `user_back` takes it
        self._seen, self._asks = None, {}   #: work together mode at the last look; the asks started or queued here, by id, until replied to
        self._stop, self._plock = threading.Event(), threading.Lock()

    def start(self):
        "Listen on `path` for this user only and return `self`. Raises `RuntimeError` when another runner listens on `path` or when `path` is not a socket."
        self.path.parent.mkdir(parents=True, exist_ok=True)
        if self.path.exists() or self.path.is_symlink():
            if not stat.S_ISSOCK(self.path.lstat().st_mode): raise RuntimeError(f'{self.path} is not a socket; nothing was removed')
            with socket.socket(socket.AF_UNIX) as probe:
                try: probe.connect(str(self.path))
                except OSError: self.path.unlink()   # a runner that crashed left it
                else: raise RuntimeError(f'another runner listens on {self.path}')
        ctl = self
        class Handler(socketserver.StreamRequestHandler):
            def handle(self):
                while (raw := self.rfile.readline(REQUEST_MAX)):
                    try: req = json.loads(raw)
                    except ValueError as e: req = e
                    out = ctl.handle(req) if isinstance(req, dict) else {'id': None, 'error': {'message': f'not a JSON request object: {req}'}}
                    self.wfile.write((json.dumps(out, default=str) + '\n').encode())
                    self.wfile.flush()
        self._server = socketserver.ThreadingUnixStreamServer(str(self.path), Handler)
        os.chmod(self.path, 0o600)   # connecting needs write access; the default umask already keeps it from others
        self._inode = self.path.stat().st_ino
        self._server.daemon_threads = True
        threading.Thread(target=self._server.serve_forever, daemon=True, name='ramabana-control').start()
        if self.agent.cfg is not None: self._peers_start()
        return self

    def close(self):
        "Stop listening and remove the socket, unless another runner has replaced it."
        self._stop.set()
        self._unhook()
        if self._server is not None: self._server.shutdown(); self._server.server_close(); self._server = None
        try:
            if self._inode is not None and self.path.stat().st_ino == self._inode: self.path.unlink()
        except OSError: pass
        self._inode = None

    def handle(self, req):
        "One request as a dict; its reply as `{'id', 'result'}` or `{'id', 'error'}`."
        rid, name, params = req.get('id'), req.get('method', ''), req.get('params') or {}
        if name not in METHODS: return {'id': rid, 'error': {'message': f'no method {name!r}; one of {", ".join(METHODS)}'}}
        try: return {'id': rid, 'result': getattr(self, f'do_{name}')(**params)}
        except Exception as e: return {'id': rid, 'error': {'message': agent_err(e)}}

    def _work(self, text):
        "Run `text`, then each prompt queued meanwhile. The last queue check and the end of running happen under one lock."
        try:
            while text is not None:
                try: self.last = self.agent.ask(text, result=True)
                except Exception as e: self.last = TurnResult(state='failed', error=agent_err(e), session=self.agent.session_id)
                self.at = time.time()
                with self._lock:
                    text = self.agent.queued.popleft() if self.agent.queued else None
                    if text is None: self._running = False
        except BaseException:   # only an error that escaped: a normal end already cleared it, and a new turn may have set it since
            with self._lock: self._running = False
            raise

    def do_prompt(self, text):
        "Start a turn on `text`, or queue it behind the one running."
        with self._lock:
            if self._running: return {'queued': self.agent.queue(text)}
            self._running = True
        self.at = time.time()
        threading.Thread(target=self._work, args=(text,), daemon=True, name='ramabana-control-turn').start()
        return {'started': True}

    def do_steer(self, text): return {'told': self.agent.steer(text)}
    def do_tell(self, run_id, text): return self.agent.tell(run_id, text)
    def do_cancel(self):
        "Stop the running turn, and drop the prompts queued behind it."
        with self._lock:
            dropped = len(self.agent.queued)
            self.agent.queued.clear()
        return self.agent.cancel() | {'dropped': dropped}
    def do_asks(self): return [a.dict() for a in self.agent.approvals.waiting] if self.agent.approvals is not None else []

    def do_answer(self, id, ok, note=''):
        ap = self.agent.approvals
        return ap is not None and ap.answer(id, bool(ok), note) is not None

    def busy(self):
        "Whether a turn runs, and how many prompts wait behind it."
        return self._running, len(self.agent.queued)

    def active_at(self):
        "When this session was last active."
        return self.at

    def do_status(self):
        running, queued = self.busy()
        roots, at, ap = list(self.agent.host.roots), self.active_at(), self.agent.approvals
        return {'running': running, 'queued': queued, 'session': self.agent.session_id,
                'last': asdict(self.last) if self.last is not None else None,
                'name': session_name(self.agent, at), 'folder': roots[0] if roots else '', 'at': at,
                'kind': 'ramabana', 'key': f'ramabana-{self.agent.session_id}', 'folders': roots, 'guard': guard_of(roots), 'approve': ap.mode if ap is not None else '',
                'state': 'waiting for approval' if ap is not None and ap.waiting else 'working' if running else 'idle'}

    def do_events(self, offset=0):
        files = self.agent.__dict__.get('_event_files') or ()
        if not files: raise RuntimeError('no event file; the runner calls agent.events_to(path) first')
        p = files[0][0]
        rotated = offset > p.stat().st_size   # the file rolled over to `.1` since the page last read it
        if rotated: offset = 0
        with open(p, 'rb') as f:
            f.seek(offset)
            data = f.read(EVENTS_MAX)
        cut = data.rfind(b'\n') + 1   # whole lines only: one still being written waits for the next read
        return {'events': [json.loads(l) for l in data[:cut].splitlines() if l.strip()], 'offset': offset + cut, 'rotated': rotated}

`start` also readies the session for messages from peers, and `close` undoes that. The methods below do the work; *Messages from peers*, at the end of this page, explains them with an example.

In [ ]:
#| export
TOGETHER_CHECK = 60   #: seconds between two looks at work together mode, to see it end
ASK_ID = re.compile(r'<peer-message id="([^"]+)" kind="ask"')
CANCELLED = 'failed: cancelled by the user'   #: the reply to an ask whose turn was stopped or dropped

def _daemon(f, name, *a): threading.Thread(target=f, args=a, daemon=True, name=name).start()

@patch(as_prop=True)
def key(self:Control):
    "This session's key in the peers folder."
    return f'ramabana-{self.agent.session_id}'

@patch(as_prop=True)
def d(self:Control):
    "The peers folder of this session's config."
    return peers_dir(self.agent.cfg)

@patch
def _peers_start(self:Control):
    "Listen for turn ends to send replies, look at work together mode every `TOGETHER_CHECK` seconds, and raise again each ask still pending."
    if self._turn_end in self.agent.turn_ends: return
    self.agent.turn_ends.append(self._turn_end)
    self._seen = together_state(self.d, self.key)
    _daemon(self._check_mode, 'ramabana-together')
    for e in self._held(): _daemon(self.receive, 'ramabana-peer', e)

@patch
def _unhook(self:Control):
    "Stop listening for turn ends."
    if self._turn_end in self.agent.turn_ends: self.agent.turn_ends.remove(self._turn_end)

@patch
def do_message(self:Control, id=''):
    "Take the new messages in this session's inbox and `receive` each on its own thread; `id` names the one that rang."
    waits = self.agent.peer_waits
    got = take(self.d, self.key, lambda e: not (e.kind == 'reply' and e.frm.get('key') in waits))
    for e in got: _daemon(self.receive, 'ramabana-peer', e)
    return {'took': len(got)}

@patch
def receive(self:Control, env):
    "Handle one peer message: a note or a reply steers the running turn or waits for the next; an ask runs in work together mode with a partner, else when the user says yes."
    a = self.agent
    if env.kind != 'ask':
        if not a.steer(peer_text(env)): a.peer_notes.append(env)
        return
    if (st := together_state(self.d, self.key)) and is_partner(st, env.frm):
        self._drop(env.id)
        if (why := limit_refusal(self.d, self.key, env)): return self._reply(env, f'refused: {why}', check=False)
        return self._run_ask(env)
    if (ap := a.approvals) is None:
        self._drop(env.id)
        return self._reply(env, 'refused: this session has no approval gate to ask its user')
    self._hold(env)
    got = ap.request('peer ask', {'from': env.frm.get('name') or env.frm.get('key', '?'), 'repo': env.repo, 'text': env.text},
                     force=True, timeout=threading.TIMEOUT_MAX)
    if ap.closed and not got.answer: return   # the session quit first; the next start raises it again
    self._drop(env.id)
    if got.answer: self._run_ask(env)
    else: self._reply(env, f"refused: {got.note or 'the user said no'}")

@patch
def _run_ask(self:Control, env):
    "Start or queue a turn on the ask `env`; `_turn_end` sends its reply."
    self._drop(env.id)
    with self._plock: self._asks[env.id] = env
    try: self.do_prompt(peer_text(env))
    except Exception as e:
        self._asks.pop(env.id, None)
        self._reply(env, f'failed: {agent_err(e)}')

@patch
def _reply(self:Control, env, text,
           check=True):   # False posts it past the limits: the one refusal a limit sends
    "Send `text` to the sender of `env` as a reply in its thread. The asker always gets a reply: one the hop limit stops goes as `failed: <why>`, and a sender with no live session, such as the `cli`, finds it in its inbox."
    me, hops = self_frm(self.agent), next_hops(self.d, env.thread, env.hops + 1)
    if check and (why := limit_refusal(self.d, me['key'], Envelope('reply', text, me, env.frm['key'], thread=env.thread, hops=hops))): text = f'failed: {why}'
    elif check:
        try: return send(env.frm['key'], text, 'reply', me, env.repo, env.thread, hops, cfg=self.agent.cfg)
        except LookupError: pass
    answer(self.d, env, me, text)

@patch
def _turn_end(self:Control, turn):
    "Reply to each ask the turn read, in its prompt or steered in: the turn's text, or `failed: <why>`. A stopped turn fails every ask still open, the dropped queued ones too."
    ids = set(ASK_ID.findall('\n'.join([turn.get('prompt') or '', *(turn.get('steered') or ())])))
    state, error = turn.get('state'), turn.get('error')
    if state == 'cancelled':
        with self._plock: ids |= set(self._asks)
    text = CANCELLED if state == 'cancelled' else f'failed: {error or state}' if error or state != 'complete' else turn.get('reply') or ''
    for i in ids:
        if (env := self._asks.pop(i, None)) is not None: _daemon(self._reply, 'ramabana-peer-reply', env, text)

@patch(as_prop=True)
def _pending(self:Control): return self.d/f'{self.key}.pending.jsonl'

@patch
def _held(self:Control):
    "The asks that wait for the user's answer."
    return [Envelope.load(r) for r in _rows(self._pending)]

@patch
def _hold(self:Control, env):
    "Add the ask `env` to the pending file, once."
    with self._plock:
        held = self._held()
        if env.id not in {e.id for e in held}: _atomic_write(self._pending, ''.join(json.dumps(e.dict()) + '\n' for e in [*held, env]))

@patch
def _drop(self:Control, id):
    "Remove the ask `id` from the pending file."
    with self._plock:
        left = [e for e in self._held() if e.id != id]
        if left: _atomic_write(self._pending, ''.join(json.dumps(e.dict()) + '\n' for e in left))
        else: self._pending.unlink(missing_ok=True)

@patch
def _check_mode(self:Control):
    "Until `close`: when a work together mode seen on has ended, save its report on a daemon thread."
    while not self._stop.wait(TOGETHER_CHECK):
        with self._lock: seen, self._seen = self._seen, together_state(self.d, self.key)
        if seen and self._seen is None: _daemon(self._save_report, 'ramabana-report', seen['started'])

@patch
def _save_report(self:Control, since):
    "Save the report of the mode that started at `since`; on a failure `user_back` tells the user to run `ramabana report`."
    try: self.report = save_report(since, self.agent.cfg)
    except Exception: self.report = None

@patch
def user_back(self:Control):
    "End work together mode, as the user typed: whether a mode ended since the user left, and the report saved when it ended, or None."
    with self._lock:
        seen, self._seen = self._seen, None
        on = together_state(self.d, self.key) is not None and together_off(self.d, self.key) is not None
    p, self.report = self.report, None
    return bool(on or seen or p), p

The example plays both sides in one process. `fake_agent` from `ramabana.testing` supplies an agent whose model replies with a scripted line. The first cell sends a prompt through the socket, waits for the turn to finish, checks how it ended and reads the turn's events.

The second cell raises an approval request for `edit_file` from another thread, as a tool would, finds it through `asks`, and approves it through the socket. The agent was built with `no_listener='wait'`. With that setting the request waits for an answer. Without it, the request is refused because no terminal is attached. An unknown method comes back as an error, and `control_call` raises it as `RuntimeError`.

In [ ]:
import tempfile, time
from ramabana import agent
from ramabana.testing import fake_agent

d = Path(tempfile.mkdtemp(dir='/tmp'))   # a Unix socket path must stay short
a, be = fake_agent(cfg=d, replies=['the first reply'], approvals=agent.Approvals(tools={'edit_file'}, timeout=5, no_listener='wait'))
a.events_to(d/'job.events')
ctl = Control(a).start()
call = lambda method, **kw: control_call(ctl.path, method, **kw)

test_eq(call('prompt', text='hello'), {'started': True})
for _ in range(200):
    if not call('status')['running']: break
    time.sleep(.01)
st = call('status')
test_eq((st['running'], st['last']['state'], st['last']['reply']), (False, 'complete', 'the first reply'))
ev = call('events')
assert {'turn_start', 'turn_end'} <= {e['kind'] for e in ev['events']} and ev['offset'] > 0
test_eq(call('events', offset=ev['offset'])['events'], [])

In [ ]:
out = {}
threading.Thread(target=lambda: out.setdefault('r', a.approvals.request('edit_file', {'path': 'a.py'})), daemon=True).start()
for _ in range(200):
    if (waiting := call('asks')): break
    time.sleep(.01)
test_eq(waiting[0]['tool'], 'edit_file')
assert call('answer', id=waiting[0]['id'], ok=True)
for _ in range(200):
    if 'r' in out: break
    time.sleep(.01)
assert out['r'], 'answered through the socket'
test_fail(lambda: call('fly'), exc=RuntimeError)
ctl.close(); a.close()

## Peers

Every terminal session listens on its own socket, so one session can reach another. `peers_of(agent)` finds the other live sessions under the same config folder. It asks each socket in `<cfg>/runs/*/` and each `ramabana-<session>.sock` of a session in that folder for its `status`, and leaves out a socket that does not answer within `PEER_WAIT` seconds. The list puts the most recently active session first.

`status` names a session for a person, the way the `--resume` picker lists sessions: its title, else its first folder's name, then its model and how long ago it was active, as in `orgs · claude-opus-5-5 · 2m ago`. `peer_lines` numbers the list, and `find_peer` takes a number from it, as the picker takes one, or a session id.

`peer_tools(agent)` gives the agent the same reach as two tools. `peers()` lists the sessions of every kind with their ids. `tell(session, text, kind='note')` posts a note to the inbox of that session, through `ramabana.peers.tell`, and rings its socket. With `kind='ask'` it asks for work and waits up to `wait` seconds for the reply. The sender of each message is `self_frm(agent)`: this session's key and all its folders.

In [ ]:
#| export
PEER_WAIT = 2   #: seconds a peer has to answer `status` before `peers_of` leaves it out

def peers_of(agent, timeout=PEER_WAIT):
    "The other live sessions under `agent`'s config folder, most recently active first: each one's `status` plus the `path` it listens on, and the registered Claude Code and Codex sessions."
    return [] if agent.cfg is None else all_peers(agent.cfg, agent.session_id, timeout)

def peer_label(row):
    "A peer's name without how long ago it was active, which a label kept on screen would leave stale."
    return (row.get('name') or row['session']).rsplit(' · ', 1)[0]

def find_peer(rows, key):
    "The row `key` names: its number in `rows`, or its session id; None when neither matches."
    return pick_row(rows, key) or next((s for s in rows if s['session'] == str(key)), None)

def self_frm(agent):
    "This session as the sender of a peer message: its kind, session, key, name and all its folders."
    roots = [str(r) for r in agent.host.roots]
    return dict(kind='ramabana', session=agent.session_id, key=f'ramabana-{agent.session_id}', name=Path(roots[0]).name if roots else 'ramabana', folders=roots)

def note_peer(agent, row, text, kind='note', wait=0):
    "Post `text` as a `note` or `ask` from this session to the session `row`, and with `wait` return the reply to an ask; what `tell` says. `LookupError` when it is gone."
    return send(row['key'], text, kind, self_frm(agent), wait=wait, cfg=agent.cfg)

def peer_tools(agent):
    "`peers()` and `tell()`: the agent's tools for reaching the other live sessions."
    shown = []
    def peers() -> str:
        "List the other live sessions, numbered, most recently active first: name, state, folder, session id and kind (ramabana, claude or codex)."
        shown[:] = peers_of(agent)
        return '\n'.join(f"{l}  {r['kind']}" for l, r in zip(peer_lines(shown, ids=True).splitlines(), shown)) or 'no other session is running'
    @acts
    def tell(
        session: str,         # a number from the last `peers()` list, or a session id
        text: str,            # what to say
        kind: str = 'note',   # `note` to tell it something, or `ask` to ask it for work
        wait: int = 60,       # seconds an ask waits for the reply; 0 returns at once
    ) -> str:
        "Send `text` to another live session as a note or an ask, and return the reply to an ask. That session's user says yes to an ask first, unless work together mode is on there."
        if kind not in ('note', 'ask'): return err(f"kind is 'note' or 'ask', not {kind!r}")
        if not text.strip(): return err('nothing to say')
        if (row := find_peer(shown, session)) is None:
            shown[:] = peers_of(agent)
            row = find_peer(shown, session)
        if row is None: return err(f'no live session {session!r}; peers() lists them')
        wait = wait if kind == 'ask' else 0
        if wait: agent.peer_waits.add(row['key'])
        try: return note_peer(agent, row, text, kind, wait)
        except (LookupError, ValueError) as e: return err(str(e))
        finally: agent.peer_waits.discard(row['key'])
    return [peers, tell]

Two more agents in the same config folder show the peer side. `listener` writes its events to `EVENT_FILE`, as a terminal session does, so a reply can be read from them. `asker` lists the sessions it can reach: the one that listens, by its folder's name since it has no title yet, and never itself.

In [ ]:
listener, _ = fake_agent(cfg=d, replies=['nothing much'])
asker, _ = fake_agent(cfg=d)
listener.events_to(listener.runs_dir/EVENT_FILE)
listening = Control(listener).start()
peers, tell = peer_tools(asker)
listed = peers()
assert listed.startswith('1  proj · fake · 0m ago  idle') and listener.session_id in listed and asker.session_id not in listed
listed

`status` also gives the `kind`, the inbox `key`, all the folders and the permission mode of the gate, so `peers`, `tell` and `together` can use it.

In [ ]:
st = control_call(listening.path, 'status')
assert (st['kind'], st['key']) == ('ramabana', f'ramabana-{listener.session_id}') and st['folders'] and 'approve' in st
st['key']

A note to an idle session waits there for its next turn. An ask starts a turn there, and `tell` returns its reply. `listener` is in work together mode, so its user is not asked first.

In [ ]:
from ramabana.peers import together_on

In [ ]:
assert tell(1, 'are you there?', wait=0).startswith('told')
together_on(peers_dir(d), f'ramabana-{listener.session_id}')
replied = tell(1, 'what are you working on?', kind='ask', wait=10)
listening.close(); listener.close(); asker.close()
assert replied == 'nothing much'
replied

## Messages from peers

A peer posts each message to this session's inbox, `ramabana-<session>.inbox.jsonl` in the peers folder, and then rings the socket with `message`. `do_message` takes the new messages and hands each one to `receive` on its own thread. A reply that a `tell` here waits for stays in the inbox for that `tell`.

`receive` handles each kind:

- A `note` or a `reply` goes into the running turn as a steer. With no turn running, it waits in `agent.peer_notes` for the next prompt.
- An `ask` runs at once when work together mode is on and the sender is a partner. It runs as `do_prompt` runs a prompt. When a limit (hops or budget) refuses it, no turn starts, and the sender gets one reply, `refused: <why>`.
- Any other `ask` goes to the approval gate as `peer ask`. Only a person answers it: no approval mode, saved rule or `/approve` switch decides it, and it waits with no time limit. While it waits, it is also in `ramabana-<session>.pending.jsonl`, so a session that quits raises it again when it starts. A no goes back to the sender as the reply `refused: <note>`.

`start` adds `_turn_end` to `agent.turn_ends`, which the agent calls with each turn's log row however the turn ended, and `close` takes it off. `_turn_end` finds every ask the turn read, in its prompt (a terminal can join queued asks into one turn) or steered in. Each ask gets the turn's text back as the reply, with its `changes:` and `proof:` lines. A failed turn replies `failed: <error>`. A stopped turn replies `failed: cancelled by the user` to every ask still open, the queued ones it dropped too. The send budget never holds back a reply, so the asker always hears back. When the hop limit stops a reply, the asker gets `failed: <why>` in its inbox; an asker with no live session, such as the `cli`, gets the reply there too.

A thread looks at work together mode every `TOGETHER_CHECK` seconds. When a mode it saw on has ended, it saves the morning report with `save_report`. `user_back` ends the mode when the user types, and gives the report to show.

In the example, a Claude Code session, `claude-s1`, sends to a session with an approval gate. A note to an idle session waits for the next turn. An ask with no work together mode raises `peer ask` at the gate, and a no goes back to the sender as `refused: <note>`.

In [ ]:
from ramabana.peers import post, register
os.environ['RAMABANA_PEERS'] = str(d/'peers')
register(peers_dir(d), 'claude', 's1', ['/r/abhyasa'], os.getpid())
frm = {'kind': 'claude', 'session': 's1', 'key': 'claude-s1', 'name': 'abhyasa'}
b, bbe = fake_agent(cfg=d, replies=['ok'], approvals=agent.Approvals(tools={'edit_file'}, no_listener='wait'))
bc = Control(b).start()
def ring(kind, text):
    post(bc.d, bc.key, Envelope(kind, text, frm, bc.key))
    return control_call(bc.path, 'message', id='')
test_eq(ring('note', 'v2 is out'), {'took': 1})
for _ in range(100):
    if b.peer_notes or b.approvals.waiting: break
    time.sleep(.02)
test_eq([e.text for e in b.peer_notes], ['v2 is out'])
ring('ask', 'add a Card part')
for _ in range(100):
    if b.approvals.waiting: break
    time.sleep(.02)
ask = b.approvals.waiting[0]
test_eq((ask.tool, ask.args['text']), ('peer ask', 'add a Card part'))
b.approvals.answer(ask.id, False, 'not now')
for _ in range(100):
    if (got := take(bc.d, 'claude-s1')): break
    time.sleep(.02)
bc.close(); b.close(); del os.environ['RAMABANA_PEERS']
test_eq([(e.kind, e.text) for e in got], [('reply', 'refused: not now')])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()